# Farm game art with FLUX (Colab), no tokens needed

Generates every image in the art manifest, in each style, from reference renders built into this notebook.
Then it removes the background, sizes each image to the game's grid, and **downloads a zip**.
Attach that zip in the chat with Claude, and it goes into the game.

| Runtime | Model |
|---|---|
| Any GPU (A100, L4 or T4) | FLUX.2 [klein] 4B (Apache 2.0, no sign-in needed) |
| A100, and an `HF_TOKEN` secret added | FLUX 3 Image Klein (better quality; accept its terms on Hugging Face first) |

**Steps:** Runtime → Change runtime type → a GPU (A100 or L4), then Runtime → Run all.
When it finishes, your browser downloads `farm_art_pilot.zip`.

In [ ]:
# 1. Settings
STYLES = ["painterly", "toon3d", "semireal"]   # the pilot tries all three; later, keep only the winner
VARIANTS = 2                                   # images per item per style, to pick the best
ONLY = []                                      # item ids to (re)make; empty = all in the manifest
OUT = "art/pilot"                              # where results go in the repo
SEED = 7
BACKEND = "auto"   # "auto", "flux3" (needs ~35 GB GPU), "flux2" (needs ~14 GB GPU) or "api"

In [ ]:
# 2. Install
!pip -q install git+https://github.com/huggingface/diffusers.git transformers accelerate sentencepiece protobuf
!pip -q install "rembg[gpu]" pillow requests

In [ ]:
# 3. Unpack the manifest and reference images built into this notebook
import os, json, base64
EMBEDDED = json.loads(r'''{"art/manifest.json": "ewogICJfbm90ZSI6ICJFdmVyeSBwaWVjZSBvZiBhcnQgdGhlIGdhbWUgbmVlZHMuIHJlZnMgPSBpbWFnZXMgKGluIGFydC8pIHRoYXQgZml4IHRoZSBhbmdsZSwgc2hhcGUgYW5kIHByb3BvcnRpb25zOyBLbGVpbiByZXN0eWxlcyB0aGVtLiB0aWxlcyA9IGZvb3RwcmludCB3aWR0aCwgdXNlZCB0byBzaXplIHRoZSByZXN1bHQuIiwKICAiY29tbW9uIjogIkEgc2luZ2xlIGlzb21ldHJpYyBnYW1lIGFzc2V0IGZvciBhIGNvenkgZmFybWluZyBnYW1lLCBzZWVuIGZyb20gMzAgZGVncmVlcyBhYm92ZSBhbmQgdHVybmVkIDQ1IGRlZ3JlZXMsIG1hdGNoaW5nIHRoZSByZWZlcmVuY2UgaW1hZ2UncyBjYW1lcmEgYW5nbGUsIG91dGxpbmUsIHByb3BvcnRpb25zIGFuZCBmb290cHJpbnQgZXhhY3RseS4gQ2VudGVyZWQsIGNvbXBsZXRlIGFuZCB1bmNyb3BwZWQsIGlzb2xhdGVkIG9uIGEgcGxhaW4gZmxhdCBsaWdodCBncmV5ICgjZThlOGU4KSBiYWNrZ3JvdW5kIHdpdGggbm90aGluZyBlbHNlIGluIHRoZSBzY2VuZTogbm8gZ3JvdW5kIGJleW9uZCBpdHMgb3duIGZvb3RwcmludCwgbm8gY2FzdCBzaGFkb3cgb24gdGhlIGJhY2tncm91bmQsIG5vIHRleHQsIG5vIGJvcmRlciwgbm8gd2F0ZXJtYXJrLiIsCiAgInN0eWxlcyI6IHsKICAgICJwYWludGVybHkiOiAiU3R5bGU6IGhhbmQtcGFpbnRlZCBjb3p5IGNhcnRvb24gZ2FtZSBhcnQsIHNvZnQgcGFpbnRlcmx5IGJydXNod29yaywgd2FybSBnb2xkZW4taG91ciBzdW5saWdodCBmcm9tIHRoZSB1cHBlciBsZWZ0LCBnZW50bGUgYW1iaWVudCBvY2NsdXNpb24sIHJpY2ggYnV0IG5hdHVyYWwgY29sb3JzLCBjcmlzcCByZWFkYWJsZSBzaWxob3VldHRlLCBwcmVtaXVtIG1vYmlsZSBnYW1lIHF1YWxpdHkuIiwKICAgICJ0b29uM2QiOiAiU3R5bGU6IHBvbGlzaGVkIHN0eWxpemVkIDNEIGNhcnRvb24gcmVuZGVyLCBzbW9vdGggcm91bmRlZCBjaHVua3kgc2hhcGVzLCBzb2Z0IGdsb2JhbCBpbGx1bWluYXRpb24gYW5kIGFtYmllbnQgb2NjbHVzaW9uLCB3YXJtIHN1bmxpZ2h0IGZyb20gdGhlIHVwcGVyIGxlZnQsIHNhdHVyYXRlZCB5ZXQgbmF0dXJhbCBjb2xvcnMsIGdsb3NzeSBzdWJ0bGUgaGlnaGxpZ2h0cywgcHJlbWl1bSBtb2JpbGUgZmFybWluZyBnYW1lIHF1YWxpdHkuIiwKICAgICJzZW1pcmVhbCI6ICJTdHlsZTogc2VtaS1yZWFsaXN0aWMgZGV0YWlsZWQgZ2FtZSBpbGx1c3RyYXRpb24sIG5hdHVyYWwgbWF0ZXJpYWxzIHdpdGggZmluZSB0ZXh0dXJlICh3ZWF0aGVyZWQgd29vZCwgc3RyYXcsIGZ1ciwgbGVhdmVzKSwgc29mdCBuYXR1cmFsIGRheWxpZ2h0IGZyb20gdGhlIHVwcGVyIGxlZnQsIHN1YnRsZSByZWFsaXNtLCByaWNoIGRldGFpbCB0aGF0IHN0aWxsIHJlYWRzIGNsZWFybHkgYXQgc21hbGwgc2l6ZSwgcHJlbWl1bSBnYW1lIGFzc2V0LiIKICB9LAogICJpdGVtcyI6IFsKICAgIHsiaWQiOiAiZmFybWhvdXNlIiwgInRpbGVzIjogMywgInJlZnMiOiBbInJlZnMvZmFybWhvdXNlLnBuZyJdLAogICAgICJzdWJqZWN0IjogImEgY2hhcm1pbmcgdHdvLXN0b3JleSByZWQgd29vZGVuIGZhcm1ob3VzZSB3aXRoIHdoaXRlIHdpbmRvdyB0cmltLCBhIGNvdmVyZWQgZnJvbnQgcG9yY2gsIGEgc2hpbmdsZWQgcm9vZiBhbmQgYSBicmljayBjaGltbmV5In0sCiAgICB7ImlkIjogImNvdyIsICJ0aWxlcyI6IDEsICJyZWZzIjogWyJyZWZzM2QvY293LnBuZyJdLAogICAgICJzdWJqZWN0IjogImEgZnJpZW5kbHkgYmxhY2stYW5kLXdoaXRlIGRhaXJ5IGNvdyBzdGFuZGluZyBjYWxtbHksIGZ1bGwgYm9keSJ9LAogICAgeyJpZCI6ICJjaGlja2VuX2Nvb3AiLCAidGlsZXMiOiAzLCAicmVmcyI6IFsicmVmcy9jaGlja2VuX2Nvb3AucG5nIl0sCiAgICAgInN1YmplY3QiOiAiYSBzbWFsbCB3b29kZW4gY2hpY2tlbiBjb29wIHJhaXNlZCBvbiBzaG9ydCBsZWdzLCB3aXRoIGEgbGl0dGxlIHJhbXAsIG5lc3RpbmcgYm94ZXMgb24gdGhlIHNpZGUsIGEgcmVkIHJvb2YgYW5kIGEgc21hbGwgd2lyZS1mZW5jZWQgcnVuIHdpdGggYSBmZXcgd2hpdGUgaGVucyJ9LAogICAgeyJpZCI6ICJhcHBsZV90cmVlIiwgInRpbGVzIjogMSwgInJlZnMiOiBbInJlZnMvYXBwbGVfdHJlZS5wbmciLCAicmVmczNkL29ha190cmVlLnBuZyJdLAogICAgICJzdWJqZWN0IjogImEgbHVzaCByb3VuZCBhcHBsZSB0cmVlIHdpdGggYSBzdHVyZHkgdHJ1bmssIGZ1bGwgZ3JlZW4gY2Fub3B5IGFuZCBwbGVudHkgb2Ygc2hpbnkgcmVkIGFwcGxlcyJ9LAogICAgeyJpZCI6ICJjcm9wX3N0cmF3YmVycmllc19zcHJvdXQiLCAidGlsZXMiOiAxLCAicmVmcyI6IFsicmVmcy9jcm9wX3N0cmF3YmVycmllc19zcHJvdXQucG5nIl0sCiAgICAgInN1YmplY3QiOiAib25lIHNxdWFyZSBwbG90IG9mIGZyZXNobHkgdGlsbGVkIGRhcmsgYnJvd24gc29pbCBpbiBuZWF0IGZ1cnJvd3Mgd2l0aCBmb3VyIHRpbnkgc3RyYXdiZXJyeSBzZWVkbGluZ3MganVzdCBzcHJvdXRpbmcifSwKICAgIHsiaWQiOiAiY3JvcF9zdHJhd2JlcnJpZXNfZ3Jvd2luZyIsICJ0aWxlcyI6IDEsICJyZWZzIjogWyJyZWZzL2Nyb3Bfc3RyYXdiZXJyaWVzX2dyb3dpbmcucG5nIl0sCiAgICAgInN1YmplY3QiOiAib25lIHNxdWFyZSBwbG90IG9mIHRpbGxlZCBicm93biBzb2lsIHdpdGggZm91ciBsZWFmeSBzdHJhd2JlcnJ5IHBsYW50cyBpbiBibG9vbSB3aXRoIHNtYWxsIHdoaXRlIGZsb3dlcnMifSwKICAgIHsiaWQiOiAiY3JvcF9zdHJhd2JlcnJpZXNfcmlwZSIsICJ0aWxlcyI6IDEsICJyZWZzIjogWyJyZWZzL2Nyb3Bfc3RyYXdiZXJyaWVzX3JpcGUucG5nIl0sCiAgICAgInN1YmplY3QiOiAib25lIHNxdWFyZSBwbG90IG9mIHRpbGxlZCBicm93biBzb2lsIHdpdGggZm91ciBsdXNoIHN0cmF3YmVycnkgcGxhbnRzIGhlYXZ5IHdpdGggcmlwZSBnbG9zc3kgcmVkIHN0cmF3YmVycmllcyJ9LAogICAgeyJpZCI6ICJoYXlfYmFsZSIsICJ0aWxlcyI6IDEsICJyZWZzIjogWyJyZWZzL2hheV9iYWxlLnBuZyJdLAogICAgICJzdWJqZWN0IjogImEgc2luZ2xlIHJlY3Rhbmd1bGFyIGdvbGRlbiBzdHJhdyBoYXkgYmFsZSB0aWVkIHdpdGggdHdvIHN0cmFuZHMgb2YgdHdpbmUifQogIF0KfQo=", "art/refs/farmhouse.png": "iVBORw0KGgoAAAANSUhEUgAAA2AAAARjCAYAAAAJoUeEAAAAAXNSR0IArs4c6QAAIABJREFUeJzs3bHLbelZxuGVYRqJf8KIsbSwELEQiRapQuxCUqpFwCaFkMZCBMXCJp2lhYJV0gqpTQT/AcFKCDjCsbJJsBFiMbPn7PN9e+9vrbXXut/nfd/ramaGMzOsamb99v2c/X3p3bt3v1gAAAA43UetHwAAAGAWAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBgAAECLAAAAAQgQYAABAiAADAAAIEWAAAAAhAgwAACBEgAEAAIQIMAAAgBABBkT881//WetHAABo7kvv3r37ReuHAMb2Mr5+/8//ptGTAAC0JcCAU/3jN7+2/Mpv/vbNXxNiAMBsBBhwqn/85te++HMhBgDMToABp7mOr2tCDACYlQADTnEvvq4JMQBgNgIMOMWaALsQYgDALAQYcLgt8XVNiAEAoxNgwKH2xtc1IQYAjEqAAYc6IsAuhBgAMBoBBhzmyPi6JsQAgFEIMOAwZwXYhRADAHonwIBDnB1f14QYANArAQY8LRlf14QYANAbAQY8rVWAXQgxAKAXAgx4Suv4uibEAIDqBBiwW6X4uibEAICqBBiwW9UAuxBiAEA1AgzYpXp8XRNiAEAVAgzYrKf4uibEAIDWBBiwWa8BdiHEAIBWBBiwSe/xdU2IAQBpAgxYbaT4uibEAIAUAQasNmqAXQgxAOBsAgxYZfT4uvj0Z/+7/M5Xf+/mrwkxAOBZAgx40yzxdU2IAQBnEGDAm2YMsAshBgAcSYABD80cX9eEGABwBAEG3CW+XhNiAMAzBBhwlwC7T4gBAHsIMOAm8fW2T3/2v8uyLEIMAFhNgAE3CbD1hBgAsJYAA14RX/sIMQDgLQIM+ID4ep4QAwDuEWDABwTYcYQYAPCSAAO+IL7OIcQAgAsBBizLIr4ShBgAIMCAZVkEWMolwpZFiAHAjAQYIL4aEGIAMCcBBpMTX20JMQCYiwCDyQmwGoQYAMxBgMHExFc9QgwAxibAYFLiqzYhBgBjEmAwKQFW33WELYsQA4ARCDCYkPjqixADgHEIMJiM+OqXEAOA/gkwmIwA658QA4B+CTCYiPgaixADgP4IMJiIABuTEAOAfggwmIT4Gp8QA4D6BBhMQHzN42WELYsQA4BKBBhMQIDNR4gBQE0CDAYnvuYmxACgFgEGAxNfXAgxAKhBgMHABBgvCTEAaEuAwaDEF48IMQBoQ4DBgMQXa9yKsGURYgBwJgEGAxJgbCHEACBHgMFgxBd7CTEAOJ8Ag4GIL44gxADgPAIMBiLAOJIQA4DjCTAYhPjiLEIMAI4jwGAA4osEIQYAzxNgMAABRsq9CFsWIQYAawgw6Jz4ogUhBgD7CDDomPiiNSEGANsIMOiYAKMKIQYA6wgw6JT4oiIhBgCPCTDolACjMiEGALcJMOiQ+KIHjyJsWYQYAHMSYNAZ8UVvhBgAvCfAoDMCjF4JMQAQYNAV8cUIhBgAMxNg0AnxxWiEGAAzEmDQCQHGqIQYADMRYNAB8cUMhBgAMxBgUJz4YiZvRdiyCDEA+ibAoDgBxoyEGACjEmBQmPhidkIMgNEIMChKfMF7QgyAUQgwKEqAwWtCDIDeCTAoSHzBY0IMgF4JMChGfMF6QgyA3ggwKEaAwXZCDIBeCDAoRHzBc4QYANUJMChEgMExhBgAVQkwKEJ8wfGEGADVCDAoQHzBuYQYAFUIMChAgEGGEAOgNQEGjYkvyBNiALQiwKAh8QVtCTEA0gQYNCTAoAYhBkCKAINGxBfUI8QAOJsAgwbEF9QmxAA4iwCDBgQY9EGIAXA0AQZh4gv6I8QAOIoAgyDxBX0TYgA8S4BBkACDMQgxAPYSYBAivmA8QgyArQQYBIgvGJsQA2AtAQYBAgzmIMQAeIsAg5OJL5iPEAPgHgEGJxJfMDchBsBLAgxOJMCAZRFiALwnwOAk4gt4SYgBIMDgJAIMuEeIAcxLgMEJxBewhhADmI8Ag4OJL2ArIQYwDwEGBxNgwF5CDGB8AgwOJL6AIwgxgHEJMDiI+AKOJsQAxiPA4CACDDiLEAMYhwCDA4gvIEGIAfRPgMGTxBeQJsQA+iXA4EkCDGhFiAH0R4DBE8QXUIEQA+iHAIOdxBdQjRADqE+AwU4CDKhKiAHUJcBgB/EF9ECIAdQjwGAj8QX0RogB1CHAYCMBBvRKiAG0J8BgA/EFjECIAbQjwGADAQaMRIgB5AkwWEl8AaMSYgA5AgxWEF/ADIQYwPkEGKwgwICZCDGA8wgweIP4AmYlxACOJ8DgAfEFIMQAjiTA4AEBBvCeEAN4ngCDO8QXwG1CDGA/AQY3iC+AtwkxgO0EGNwgwADWE2IA6wkweEF8AewjxADeJsDgivgCeJ4QA7hPgMEVAQZwHCEG8JoAg8+JL4BzCDGA9wQYLOILIEGIAQgwWJZFgAEkCTFgZgKM6YkvgDaEGDAjAcbUxBdAe0IMmIkAY2oCDKAOIQbMQIAxLfEFUJMQA0YmwJiWAAOoTYgBIxJgTEl8AfRDiAEjEWBMR3wB9EmIASMQYExHgAH0TYgBPRNgTEV8AYxDiAE9EmBMQ3wBjEmIAT0RYExDgAGMTYgBPRBgTEF8AcxDiAGVfdT6AeBs4gtgLp/88i8tn/zyLz38e/71Jz9e/vUnPw49EcB7AgwAGNLWEPvot359+cmP/iHxaMDEnCAyNOsXABdvnSb+7p/+yQd//dWv/9GZjwNMSoAxLPEFwC23QuxlfF0TYsCRnCACAFN5eZr4KL6WZXGWCBzq49YPAGewfgHwlrd+f9i1S4RZw4BnOUFkOOILgLV+9Tt/uOufE2LAXk4QAYAp7Y2vZflsEXOaCOwhwBiK9QuAJCEGbCXAAIDpPLN+3SLEgLUEGMOwfgHQmhAD3iLAGIL4AmCto9evW0QYcI8AAwCmkYivC2sYcIsAo3vWLwAqE2LANT8HjK6JLwDWSq5fj/gZYjA3CxgAMLwq8bUsFjGYnQCjW9YvAHomxGBOAowuiS8A1qq0ft0iwmAuAgwAGFb1+LqwhsE8BBjdsX4BMCohBuPzLYh0RXwBsFYv69cjvjERxmMBAwCGM0J8LYtFDEYkwOiG9QuAWQkxGIcTRLogvgBYa5T16xGnidAvCxgAMIwZ4mtZfHU99EyAUZ71CwBec5YIfXKCSGniC4C1Zlm/7nGWCH2wgAEA3Zs9vpbFIga9EGCUZf0CgO2EGNQmwACArlm/bhNiUJMAoyTrFwAcQ4RBLb6Eg3LEFwBrWb+28UUd0J4FDADokvjazlkitCfAKMX6BQDnE2LQjhNEyhBfAKxl/TqW00TIsYABAF0RX8eziEGOAKME6xcAtCfE4HwCjObEFwBrWb8yRBicR4ABAF0QX1nWMDiHAKMp6xcA1CbE4Fi+BZFmxBcAa1m/6vCNifAcCxgAUJr4qsUiBs8RYDRh/QKAvgkx2EeAESe+AFjL+lWfCINtBBgAUJL46oc1DNYTYERZvwBgXEIM3ibAAIByrF99E2JwnwAjxvoFAHMRYvCanwNGhPgCYC3r17j8DDGwgAEAhYivsVnEQIARYP0CAK6JMGbmBJFTiS8A1rJ+zclZIrOxgAEAzYmveTlLZDYCjNNYvwCAtYQYs3CCyCnEFwBrWb+4xWkio7KAAQDNiC/usYgxKgHG4axfAMBRRBijcYLIocQXAGtZv9jKWSIjsIABAHHiiz2cJTICAcZhrF8AQIIQo2dOEDmE+AJgLesXR3OaSE8sYABAjPjiDBYxeiLAeJr1CwCoQIjRAwHGU8QXAGtZv0gRYVQmwACA04kv0qxhVCXA2M36BQBUJ8SoRoABAKeyflGBEKMKAcYu1i8AoEdCjNb8HDA2E18ArGX9ojo/Q4w0CxgAcArxRQ+sYaQJMDaxfgEAo3GWSJITRFYTXwCsZf2iZ84SOZMFDAA4lPiidxYxziTAWMX6BQDMRohxBieIvEl8AbCW9YuROU3kCBYwAOAQ4ovRWcQ4ggDjIesXAMCHRBjPcILIXeILgLWsX8zKWSJbWcAAgKeIL2bmLJGtBBg3Wb8AANYTYqzlBJFXxBcAa1m/4DanidxjAQMAdhFfcJ9FjHsEGB/4n5/+S+tHAAAYhgjjJQHGK9/4/l8u3/j+X7Z+DAAKs34B7CPAuEuIAQDAsQQYbxJiAFyzfgHsJ8BYTYgBIL4AniPA2EyIAQDAPgKM3YQYwFysXwDPE2A8TYgBjE98ARxDgHEYIQYAAI8JMA4nxADGYv0COI4A4zRCDKB/4gvgWAKM0wkxAAD4jAAjRogB9MX6BXA8AUacEAOoT3wBnEOA0YwQAwBgNgKM5oQYQC3WL4DzCDDKEGIA7YkvgHMJMMoRYgAAjEqAUZYQA8iyfgGcT4BRnhADOJ/4AsgQYHRDiAEA0DsBRneEGMCxrF8AOQKMbgkxAAB6I8DonhAD2M/6BZAlwBiGEAPYRnwB5AkwhiPEAACoSoAxLCEGcJ/1C6ANAcbwhBjAh8QXQDsCjGkIMQAAWhNgTEeIATOzfgG0JcCYlhADZiO+ANoTYExPiAEAkCLA4HNCDBiZ9QugBgEGLwgxYDTiC6AOAQZ3CDEAAI4mwOANQgzomfULoBYBBisJMaA34gugHgEGGwkxAAD2EmCwkxADKrN+AdQkwOBJQgwAgLUEGBxEiAFVWL8A6hJgcDAhBrQkvgBqE2BwEiEGAMBLAgxOJsSAFOsXQH0CDEKEGHAm8QXQBwEGYUIMAGBeAgwaEWLAUaxfAP0QYNCYEAOeIb4A+iLAoAghBgAwPgEGxQgxYC3rF0B/BBgUJcSAR8QXQJ8EGBQnxAAAxiHAoBNCDLiwfgH0S4BBZ4QYzE18AfRNgEGnhBgAQH8EGHROiME8rF8A/RNgMAghBmMTXwBjEGAwGCEGAFCXAINBCTEYh/ULYBwCDAYnxAAA6hBgMAkhBn2yfgGMRYDBZIQY9EN8AYxHgMGkhBgAQJ4Ag8kJMajJ+gUwJgEGLMsixKAS8QUwLgEGfECIAQCcR4ABNwkxaMP6BTA2AQY8JMQgR3wBjE+AAasIMQCA5wkwYBMhBuewfgHMQYABuwgxOI74ApiHAAOeIsQAANYTYMAhhBjsY/0CmIsAAw4lxGA98QUwHwEGnEKIAQC8JsCAUwkxuM36BTAnAQZECDEAAAEGhAkxsH4BzEyAAU0IMWYlvgDmJsCApoQYADATAQaUIMSYgfULAAEGlCLEGJX4AmBZBBhQlBADAEYkwIDShBgjsH4BcCHAgC4IMXolvgC4JsCArggxAKBnAgzokhCjB9YvAF4SYEDXhBhViS8AbhFgwBCEGADQAwEGDEWIUYH1C4B7BBgwJCFGK+ILgEcEGDA0IQYAVCLAgCkIMRKsXwC8RYABUxFinEV8AbCGAAOmJMQAgBYEGDA1IcYRrF8ArCXAABYhBgBkCDCAK0KMraxfAGwhwABuEGKsIb4A2EqAATwgxACAIwkwgBWEGC9ZvwDYQ4ABbCDEWBbxBcB+AgxgByEGAOwhwACeIMTmY/0C4BkCDOAAQmwO4guAZwkwgAMJMQDgEQEGcAIhNh7rFwBHEGAAJxJiYxBfABxFgAEECDEAYFkEGECUEOuP9QuAIwkwgAaEWB/EFwBHE2AADQkxAJiLAAMoQIjVY/0C4AwCDKAQIQYAYxNgAAUJsbasXwCcRYABFCbE8sQXAGcSYAAdEGIAMAYBBtARIXYu6xcAZxNgAB0SYscTXwAkCDCAjgkxAOiLAAMYgBB7jvULgBQBBjAQIbad+AIgSYABDEiIAUBNAgxgYELsMesXAGkCDGACQuw18QVACwIMYCJCDADaEmAAE5o9xKxfALQiwAAmNmOIiS8AWhJgAEwZYgDQggAD4Aujh5j1C4DWBBgAr4wYYuILgAoEGAB3jRhiANCSAAPgTb2HmPULgCoEGACr9R5iANCaAANgs55CzPoFQCUCDDr2ve98t/UjMLnqISa+AKhGgEHnvved7woxmqseYgBQhQCDQQgxKqgUYtYvACoSYDAYIUYFrUNMfAFQlQCDQQkxKmgdYgBQjQCDwQkxKkiGmPULgMoEGExCiFHB2SEmvgCoToDBZIQYFThNBGBWAgwmJcSo4MgQs34B0AMBBpMTYlTwbIiJLwB6IcCAZVmEGDU4TQRgdAIM+IAQo4ItIWb9AqAnAgy4SYhRwVshJr4A6I0AAx4SYlTgNBGAUQgwYBUhRgXXIWb9AqBHAgzYRIhRgTUMgF4JMGAXIQYAsJ0AA54ixAAA1hNgwCGEGADA2wQYcCghBgBwnwADTiHEAABeE2DAqYQYAMB7AgyIEGIAAAIMCBNiAMDMBBjQhBADAGYkwICmhBgAMBMBBpQgxACAGQgwoBQhBgCMTIABJQkxAGBEAgwoTYgBACMRYHzh008/bf0IcJcQA6BX3rG4JsCArggxAKBnAoxlWXwyQ3+EGAA98a7FhQADuibEAICeCDBgCEIMAOiBAMMkzlCEGABVeediWQQYMCghBgBUJMAm55MYRifEAKjEuxcCDJiCEAMAKhBgwFSEGADQkgCbmAmcmQkxAFrxDjY3AQZMTYgBAEkCbFI+eYEPiTAAkryLzUuAAQAAhAgwAACAEAE2IZM3AEB73snmJMAAAABCBNhkfNICAFCHd7P5CDAAAIAQAQYAABAiwCZi4gYAqMc72lwEGAAAQIgAm4RPVgAA6vKuNg8BBgAAECLAAAAAQgTYBEzaAAD1eWebgwADAAAIEWCD80kKAEA/vLuNT4ABAACECDAAAIAQATYwEzYAQH+8w41NgAEAAIQIsEH55AQAoF/e5cYlwAAAAEIEGAAAQIgAG5DJGgCgf97pxiTAoGM/+NEPlx/86IetHwMAgJUE2GB8UjInIQYAY/JuNx4BBh379te/tXz769/64q+FGABAbQIMBiDEAAD6IMAGYqJGiAHAeLzjjUWAwYCEGABATQJsED4Z4RYhBgBj8K43DgEGExBiAAA1CDCYiBADAGhLgA3AJM1WQgwA+uOdbwwCDCYmxAAAsgRY53wSwhGEGAD0wbtf/wQY8AUhBgBwLgHGB37+8VdaPwIFCDEAOMav/cbXWj8CxXzc+gHY76wJ+hJhX/6/n57y76cflwi7xNflj9dxBgC8dmZ4ffrpp8snn3xy2r+fcwkw7hJiXAgxAFjH4sVbBFinkr8BU4hxIcQA4LZ0eFnB+iXAWE2IcSHEAOAzFi+2EmBsJsS4EGIAzEp4sZcAYzchxoUQA2AWwotnCTCeJsS4EGIAjEp4cRQBxmGEGBdCDIBRCC+OJsA69cknn0S/CXELIcaFEAOgV9XDyzcg9kuAcRohxoUQA6AX1cOL/gmwjlVewa4JMS6EGABV9RRe1q++CTBihBgXQgyAKnoKL8YgwIgTYlwIMQBaEV60IsA618sZ4i1CjAshBkBK7+Hl/LB/AozmhBgXQgyAs/QeXoxDgA2g5xXsmhDjQogBcJSRwsv6NQYBRjlCjAshBsBeI4UXYxFglCXEuBBiAKwlvKhOgA1ilDPEW4QYF0IMgHtGDy/nh+MQYHRDiHEhxAC4GD28GI8AG8jIK9g1IcaFEAOY10zhZf0aiwCjW0KMCyEGMI+ZwosxCTC6J8S4EGIA4xJejEKADWaWM8RbhBgXQgxgHLOHl/PD8QgwhiPEuBBiAP2aPbwYlwAb0Mwr2DUhxoUQA+iH8HrP+jUmAcbwhBgXQgygLuHFLAQY0xBiXNxeFG3MAAAJ2klEQVQLsf/8r/9u9kwAsxJezEaADcoZ4n1CjIuXIQZAjvB6zPnhuAQY0xJiXFxC7Pt/97eNnwRgfMKL2QmwgVnB1hFiAHA+4bWe9WtsAgw+J8QA4HjCCz4kwOAFIQYAzxNecJsAG5wzxP2EGABsJ7ye4/xwfAIM3iDEAOBtwgvWEWATsIIdQ4gBwGvC6zjWrzkIMNhIiAGA8IK9BBjsJMQAmJHwgucIsEk4QzyPEANgBsLrXM4P5yHA4CBCDIARCS84lgCbiBUsQ4gBMALhlWP9mosAg5MIMQB6JLzgXAIMTibEAOiB8IIMATYZZ4jtCDEAKhJebTk/nI8AgzAhBkAFwgvaEGATsoLVIMQAaEF41WH9mpMAg8aEGAAJwgtqEGBQhBAD4AzCC2oRYJNyhliXEAPgCMKrNueH8xJgUJQQA2AP4QW1CbCJWcH6IMQAWEN49cP6NTcBBp0QYgDcIrygLwIMOiPEAFgW4QW9+qj1A9CWCbxfP//4K8tf/f0/tX4MAMJ+7Te+Jr465t0LCxh07hJhf/HHf9D4SQA4k+iCMQgwfBnHIIQYwJiE1zisXyyLAIPhCDGAMQgvGJMAg0EJMYA+CS8YmwBjWRZniCMTYgB9EF5jc37IhQCDSQgxgJqEF8xFgPEFK9gchBhADcJrHtYvrgkwmJQQA2hDeMHcBBhMTogBZAgvYFkEGC84Q5yXEAM4h/Cam/NDXhJgwAeEGMAxhBdwiwDjFSsYyyLEAPYSXlxYv7hFgAEPCTGAdYQXsIYA4yYrGC8JMYDbhBe3WL+4R4BxlwjjFiEG8BnhxT3ii0cEGA9d/gMixHhJiAGzEl7cI7xYQ4CxijWMe4QYMAvhxSPii7UEGKuJMB4RYsCohBdvEV9sIcDYxEkibxFiwCiEF28RXuwhwNjFGsZbhBjQK+HFGuKLvQQYu4kw1hBiQC+EF2uJL54hwHiKk0TWEmJAVcKLtYQXRxBgHMIaxlpCDKhCeLGF+OIoAozDiDC2EGJAK8KLrcQXRxJgHMpJIlsJMSBFeLGV8OIMAoxTWMPYSogBZxFe7CG+OIsA4zQijD2EGHAU4cVe4oszCTBO5SSRvYQYsJfwYi/hRYIAI8Iaxl5CDFhLePEM8UWKACNGhPEMIQbcI7x4lvgiSYAR5SSRZwkx4EJ48SzhRQsCjCasYTxLiMG8hBdHEF+0IsBoRoRxBCEG8xBeHEV80ZIAoykniRxFiMG4hBdHEV5UIMAowRrGUYQYjEN4cSTxRRUCjDJEGEcSYtAv4cXRxBeVCDBKcZLI0YQY9EN4cTThRUUCjJKsYRxNiEFdwosziC+qEmCUJcI4gxCDOoQXZxFfVCbAKM1JImcRYtCO8OIswoseCDC6YA3jLEIMcoQXZxJf9OJL7969+0Xrh4C1RBhn+/L//bT1I7DSv/37f7R+BFYSXpxNfNETCxhdcZLI2X7+8VeWZRFicAThxdmEFz0SYHTJSSJnE2Kwn/AiQXzRKwFGt0QYCUIM1hNepIgveibA6JqTRFKEGNwnvEgRXoxAgDEEaxgpQgzeE14kiS9GIcAYhggjSYgxM+FFmvhiJAKMoThJJE2IMRPhRZrwYkQCjCFZw0gTYoxMeNGC+GJUAoxhiTBaEGKMRHjRivhiZAKMoTlJpBUhRs+EF60IL2YgwJiCNYxWhBg9EV60JL6YhQBjGiKMloQYlQkvWhNfzESAMRUnibQmxKhEeNGa8GJGAowpWcNoTYjRkvCiAvHFrAQY0xJhVCDESBJeVCG+mJkAY2pOEqlCiHEm4UUVwgsEGCzLYg2jDiHGkYQXlYgv+IwAg89Zw6hEiPEM4UUlwgs+JMDgBWsYlQgxthBeVCO+4DUBBjeIMKoRYjwivKhIfMFtAgzucJJIRUKMa8KLioQXPCbA4A3WMCoSYnMTXlQlvuBtAgxWEGFUJcTmIryoTHzBOgIMVnKSSGVCbGzCi8qEF2wjwGCj6//RiDGqEWJjEV5UJbpgPwEGTxBjVCXE+ia8qEh0wTEEGBxEjFGREOuL8KIa0QXHE2BwAjFGNUKsNuFFJaILziXA4GRijEqEWC3CiypEF+QIMAgSY1QhxNoSXlQguqANAQaNiDEqEGJZwovWRBe0J8CgADFGa0LsXMKLlkQX1CLAoBgxRktC7FjCi1ZEF9QlwKAwMUYrQuw5wosWRBf0QYBBJ8QYLQixbYQXaaIL+iPAoENijDQh9pjwIkl0Qd8EGHROjJEkxD4kvEgRXTAOAQYDEWOkzB5iwosE0QVjEmAwKDFGwmwhJrw4m+iC8QkwmIAY42yjh5jw4kyiC+YiwGAyYowzjRZiwouziC6YlwCDiYkxztJ7iAkvziC6gGURYMDnxBhn6C3EhBdHE13ASwIMeEWMcbTqISa8OJLoAh4RYMBDYowjVQsx4cVRRBewlgADVhNjHKV1iAkvjiC6gD0EGLCLGOMI6RATXjxLdAHPEmDA08QYzzo7xIQXzxBdwJEEGHAoMcYzjg4x4cVeogs4iwADTiPG2OvZEBNe7CG6gAQBBkSIMfbYGmLCi61EF5AmwIA4McZWb4WY8GIL0QW0JMCApsQYW7wMMeHFWqILqOJL7969+0XrhwB4SYwBzxJdQEUCDChPjAFriS6gOgEGdEWMAS+JLqAnAgzolhiDeYkuoFcCDBiCGIPxiS5gBAIMGJ44g36ILGB0Agzgc0INziOsAD4jwAA2EmrwnrAC2EaAAZxEqNEzYQVwDgEG0JhQI0lYAbQlwAA6IdR4RFgB9EGAAQxGqI1FWAGMRYABTEqotSWsAOYkwAB4SKhtI6wAeESAAXCI0UNNWAFwBAEGQFS1UBNWACQJMABKejbUhBUAFQkwAACAkI9aPwAAAMAsBBgAAECIAAMAAAgRYAAAACECDAAAIESAAQAAhAgwAACAEAEGAAAQIsAAAABCBBgAAECIAAMAAAgRYAAAACECDAAAIESAAQAAhAgwAACAEAEGAAAQIsAAAABCBBgAAECIAAMAAAgRYAAAACECDAAAIESAAQAAhAgwAACAEAEGAAAQIsAAAABCBBgAAECIAAMAAAgRYAAAACECDAAAIESAAQAAhAgwAACAEAEGAAAQIsAAAABCBBgAAECIAAMAAAgRYAAAACECDAAAIESAAQAAhAgwAACAEAEGAAAQIsAAAABCBBgAAPD/7dexAAAAAMAgf+tZ7CqLmAgYAADARMAAAAAmAgYAADARMAAAgImAAQAATAQMAABgImAAAAATAQMAAJgIGAAAwETAAAAAJgIGAAAwETAAAICJgAEAAEwEDAAAYCJgAAAAEwEDAACYCBgAAMBEwAAAACYCBgAAMBEwAACAiYABAABMBAwAAGASLT7uMOY35UkAAAAASUVORK5CYII=", "art/refs3d/cow.png": "iVBORw0KGgoAAAANSUhEUgAAAHwAAABzCAYAAABaSGZrAAAAAXNSR0IArs4c6QAAGJpJREFUeJztnXlwG9d9x7+Lk5QBAqSig6J46LQlk5KvWIdtYplmMp02tdOxEzVOYjtN0zpOm6RpJlcjAUwz006aZurmcCxHpmzJro+omcqatqNIISTHkixLFCmRlkWLl0SKNwACILHYxQL9Y/EWu4sFsDhIUBQ/Y5nY+5Ff/H7v937vWAqL3LTQNE0Hp2dw333baLLv3o/e73j+uZ81n3vvrFvtGsNcFW6RwkHTNP13X/ums2HrPTQAMAwDnufF455dX0Aqwak5KeEiaaHjAIDb7XZLj0m3lUJLkYru8frw5b/8QlNvT7dbed6i4EWEpmna6XQ6idipCDFhhEIsJr1+1eNn3z3t/tmzP22+b3ujc8PGTbRer0NX50X3r37+kybluYsuvQhoFTrMRhAIhhCcDoHjuKTjpaVmPPbnDze9++5pNwB4vF7cselOGgDu2FSvem9dnmVfJAtomqZb46QTO8xGMOEJYMLjR5jlYDQasGRJKUpKzDAajYhFo4hwHKZ8frz2+qHWZ776dRcALF+2DMuXfQQAoNfrsGHjHUnPWLTwOSAriw6EEGbj1qyocHU6HXQ6HYxGA4KBIKLRKKZnQnjmq19zAkBHe5t75YplsFotAADLbbclPWOxDp9FtAvNyYVWyqKiUiQSQTAQlO0zmUzgeR7jExMAgJYX9za37Nvrkp6zaOGzQDZC+2VCE21jkKkckx0EABgMBlisFpnoLMsiEomkLdui4AUkH6EJCalVVFbsMhgMKCktARNixFMMBgMqyssRCARUn70oeAGw28vpNWvXOO+++x768OG3cPjwW0nnbL1rK6I8j08+/CjCYS5tZSrXNb3wJSUlACATHQCsVitKS0qT7r1Yh+eB3V5Or1mzxmm3l9PpzotGeXAcB57ncWd9A5p/+M/CAY1/fSrNFtlkGCZJdAD45S/+o/mXv3jWRbb12h65CMFut9ObNm1u2bRp0/7KysqnSkpK6pLPSojChBmwHIdYTDDL8bExdHVdQlPTx5WnpiWt6ACC09Pw+nzQ63TgeV78d8+999E8H6Xa2s65gUXBNUOEXrNmjUtdZDkcx4EJM6LQQEKmQove138Nw8MjMJlNCLMseJ5HNBYTRQcF/M+Rwy9l8ahbE7vdTtfW1jnLy4nLpkBl+Ivx0ajovlNBbpGve+/rv4a+/gEAFGqqq2Tn6fQ66CgKer1g01/9ypea2s6fcy8GbSoQoe12Ow1AYqUxkI+C8OTPTyEayyw0gUTiXZ2X4NzzPUF0ldgs1bU+3xT6+gbgm5oCANhsZUnnRfkooirXL1p4HKXI2cDzPCIRHjHEZC48E6qWLj2gwOubQn/fALw+H4i3sdnKYCtLFlzJH94+6f7B9/+h6Za38EIITaBAARQ0i65q6dIDcbw+H/p6B+D1CRZNqpVYDJrE7rrcjYHrQ/Ey3oLkIzIARKNCQBSNqjlNFMzSvT4fevsG4PX6ZFUIEdxus6m6c4LfH0DX5W5x+3vfepq65XrLauvqXFu23tVqs9vpGBJVpxai0Rg4LgKO41KKDQiWTmWK7iSQMhBLB4Devn6cb2uH1+tTvcZmK0OZzRq/Ovm36LrcLRObcMu4dLvdTm/Zeler2jHln0spVSaLVoOIHo1pu4Z48XfeeQefe/wv8NSXnpa1CITj8f/HgJISs6LswqeAP6gqNGHBC07cty0L9y3G5LEYotEo+EjmyDsVOkqHWCyGWAZf4pvyYypeR5MvViwGUFT8qyCp1+22MphNZvk3lQLev9wNv1/egyZl7bqN9IIVPBehCbGYwqKJpWXj/yUQ965Wr0uFlnLqHTd2PkCrlS6p3vYHghgcupFWbMKCFPzhRz7leuSRTznb29vR0dGu+bokoZVQyF10RQSfSmgAMBoMOPUHQXCplcdiQLndJjt3cGgYg0PDmsuxoAR3OGj62O9bxXr6iSefFI91dLSjo71D3JZ+GTIKLSVP0X1TU/ClEFrJ9Wv9qK6pE7ft9kRUno1VE9au20gviGaZw0HTu51Op8ORvh9aCcdFMD3D4Hvf+RY6Oy9l/2CNwjMMA4ZhxHZ0OoxGA8qsVgBAdW0ddn32KQjOgUJdbTWA7K2acPzokeabuvPE4aDpfS0tLXucLlddXV2dlms4LoJAcAaB4AyYMItoNIqmj30cXZ2XMDY2ll0BMpgLwzAYn5iEzzcFhglruqVer4PZLETgfr8P1TV1sNnsKC+3Icxx6O0bwPiEJ7tySrgp2+EOB00fO97a+rvjra2NjTStJcfBcRH4poLwTQXBccnDgH74o39BfX1D9oWhkCQ8wzAYHhnF8MgoGCa5jzodBoMxsRETgje73YZgcAaXL3+YlQtX46aycIeDpve92NKye4/LVavBoikqYdEzM0zGOjpnSwcASm7RmcaWpcJoNMJoTIRWfr8PU1MBmEutOd1Pitc72X9TCJ6t0ADARSIITocwEwojGtUeZS1fvgKtvz+WVfkYhsH4eH5CE5SCA0Bn50UAFFZX1+Z17/KKpXXzXvDde1yuF/a1tNTW1tVpyVZykQimp0MIhcIyi9aa6ly+fAXq67dkFJ2IPDExiWBwOm+hCUuWlEKvS9S0XCSCcJjF4HWh3ztf0ed1lP67Y62tjWkib6mGXCSCUCisWj9L0dqp0dV5Cbt/8F3ZPoZh4PVOiZ9ng6UV5bLtmRCDUCgk2/fYridyEr63p9s9Ly280UHTv97X0pJObEKE5xEKhcEwrKbBBxSlrWNDaulKl10oa1ZjSal8pCmn8rz3uzqQi7W3nTv90rxLvDQ6aPp3xxLJk1REeB7hMCv2R1MUBZ1Op7mDg6KotNY+MjqGwRsTWFFZhzOnTmgsff4ousKTrJtAyrR9Z2NW959Xgu/e43L9YLfTmem8cJgFE2aT9hPRYzFt/dFK0UdGxzA6Oo6RkTGMjAqR+uD1/ix+g/xQBmuZOHPqRFaC9/Z0F29MW2Oj4K4fcjjoxkbaAQAOh4OOxWIpXW4qoaUQl016ujJBURRGRsbQ3tEpilxsiJVzGaqObF36nAne2EjT/yix3IcaHbTaeWLrKRYDRcVdG0WB53lMT6u7tlRkcvEjo3GRR+aHyIAi6QJkDEBXV9dl/4xMJ1isVtpqsdIAMDx8w6X1xkTkVOISWJaVfOYQDrNJx8xmE0wmk9ZHi1DxYbrE2onIADQLLTSH5h4mHEY4rC0dqwWy/EdawVeurHSZzCVOoScphs31Dc7rA/1NgUDArTy30UHTjY00/VCjw5FJZI6LIDQTin/mwEejaSPscJhFOMzmJPzwyChGRsZwY3hkXllzKiKRCKanZ4SNAjaa+3q6TwAZBA8Gg+71lZXOSCQS75elsH7DxtbR0ZHmG0NDLgdN07v3OJ0PPpjcfIohhvh/omsKzagvXUESDZmaVcT604lOBB4eFnLZNxM8HwHLhiEqnSHuzKUtnkHwgHt0dKS5alWVk6J08b7gGGpr6pwHDh50SLsjYzGIg/QZhgPLsmDj02EjkUjGtq9eJ8yUINNkUkGaYmazCXq9flYFnmt3HuF59fnhKchGcE0uHQCGb9xwlVnLHGSG5N333o2/efppNDQ00NLzKEoYi8WyPFiWRTA4LbuPTqfLKDpFUejoaAfP8zh//py4/0LbedXzGz/2J5mKf3MR94jCnyl1awXILUIHNLbDr1z5oGnVqirXHqfT+Vdf/rLqOTwfRSgkWLTBYIDdbgPDhMUUZDQaFSNnJefPn8Pe558TP2tl7YY7884tp6NYAVssbuCUik9fXV2L7TsdOf/emgSvqKigv/2d7zhSiS1YdXITggylJaKToUQ6nQ5tbeex9/nnshJ4IbN9p0PsG3j39NuCtVOJ5FA+QksX6MsouM1mp4+3ulsbGpIHB/C84MJ5PnWCg4h+6tQ7eGHv8wCAtrbCiDx4fWBWLXyu2L7TIcuYbdvRiHdPnwQg/I7bdjTm9XuSCB3QIPjjn/t865o1G8GyPPR6Cno9iagTLjwd7e0X8POfPYu333475wKnQkh7Zk4tDl4fwJlTJzB4fUC0kvnyRUnV87VtR2P8Z2Gfl1bwmppaV0PDVgAQXbbJZADHRaCll7G9/QKeeeavMTnhgdlskiVV5gKp0ARpR0gm8Wczj55vXZwNWbl0JSwbQTgchsFgSBmEAcDff+Nv0d5+AZPxAXcmkxkfWboUM6GQ2N0XCOQ3PitVUHXm1ElNPVzSc77xrd15lSUbVlfX4rFdT8zZ8zQLbrPZHQ1btsr2kdw0Ec1gMIirDACCVb+0f58g9mRidGUsPsfKYDQgwnFgOQ7l5TbMzITysnxSj6tZc77MRpSurK9nG+WKyhkEt9ENDVtk+5TdjmRAgMFgwMEDL2H//n0ABOuVChkOs2BZFjHEYDKbYY4vN2UvtyM0E4Lfr76uWCZmQ+jZYq7FViPlMGWbTX1OVqrRHi+/1IKXX24BIIit5q5ZjgVFUTAY9DAY9DAaDdBROlgst6GsLLdRmQtdbLU1z/MhjeA2umGL3LrT9S+/+soB6CidMCxYRWyO4zA1FcD4+CQYJgyTyQSDwSB0iBiNKDGbsWRJ8kJyxaLQX6JcxD5+9EjzC8/9VHWhe62sXbeRlm6nnYhAInRCqlEkr75yQPysxTVPTfnh9U0JQR9FQafTw2w2ZbVqQqGZTS+xfacjp+uOHT3iAgThC1WWlILX1NQmDTVSs/BXXzkgCj7l92vuww34AxgZGcP4+AQCwQBGx8YRUllJ8GYna1fu8QIej8yV9/Z0uwsluqrgpP5OFaFLkYo9NaX+ioZUsCyLQHAa4+OTYs/afKFQFp+12GffAzxeWXYMEKw9V9cudespBLfRACCN0NX6qqWuPFux5xvKBEghEiJZufKrPYLYAODxqgZrhbDyvCYTSq37ZkZNmHzTr1m58rPnBMEJCpdO6O3pducbtasKXlNT4/zCE0/K9imbY/m48vlGKmFzDbaEazWI7fEKYnvk0397vRPuVJfkErWndek1NbUuALjnnnuwZIlZaCvr5B3xskBtAYidSvBcrVzTF4W4cIXYWL8OfUvL0uaFs3Xta9ZtFAuUlGmrqalxPvnUF7H1rrsAAEajHhSlg8kUFVfnXSiuHMg81PexXU9g8PpA/F9/xmBOiyuvHBvHsNSFA0BFBbB+LVBRgWO/+ldXuut7e7rdj/zZH+NK91V8cOVq2mcBabpHiXU/+dQXxX3RqDCLgwwcbHnxBQALw5UD2lxvwtKFc8+cOhn/mWyI6e5nmZ5G5eg4LNPTCJjNCJImbEUFcP99ALKz3ts3rsftG9drFh5QsXCp2ADA8/JkyH++ehAmk3GBiJ1bHU1E3b6zUbT+TFimp7Ghtz/5wPp1wr88IMK/c+osJiaTlwMhCRxAIrjNZqN373E6ldYt5cDL+wEI77bMBtKbNd8oRNNLSz1vuT6IytExSBPOdfWbYXtwB65MeGQiScWZDSSC2+lM1n3g5RZ4vD54PF7NDyAjOrT2Uc8VczHqpben242rPYDHi4vxffffL7z6eecXPw8AmJC44nybXGrWrawiRMF375HP2kxp3TmIDSTc4HwRPZd5Wdlw/OiRZjVrfeDrX2n96P3babVrlNm1dLx39ow71X3SYQCEabpKV65m3UM3tK8N5vVMNF25fIleXZ3Iyc+G6GSo0JatW1G1qhKhUAgfdl8BAHz4Yfxn9wfxn1fE62azXzqV2EomFRY5G+5ceU8DADz88CMK65ZfdODl/fB4fZo6N1g27J4OBppZNuw+dvSIGwD+6BOfLKjoRGTyGRCWylhaUY7BoRA2bLwdAMSf+NOHZddf7OjAbEA6ObS65onJhLecDXeudk/Ddx2fcG2eSkzFJc0wKXuff06TK58OBpqDQb9Luo98wwoheqaBf6WlJVhaUY7JDGXdsPF2DN0YyerZmdBq1VeufICP3r89aX827lwNpbdIdU/Dg7XrHdyJ0+BOnIbRsQORSBSGxsTYWMG60/8BpVatdvzY0SOuzfVbHJWramiyT6vo2Y7urKgoV10IZzaQdltqtdCTJ35P22zCArlLl60Wh2hd6jiv6fpsy6fcZ3hv8BoqrXasW7oUYfcpsQuUiO7xetO6cjWrVuP40SPNX//m9+lJT6JJl050MqpTs9DlidWPiGufDXIRmUASW28d/i0A4OrVXvFYKmPJB1XB3+xsa968fBV9dXICaysqsKa8AtzJU+BOnsKRcADPHnxR9WaZrFpJV+dF9/DQNffadRvpQYk7VRM93/HapaUlKC0tTWnloSyX3MpHZEJ1dbWrpqbGCQgTBqVV5HQwkHe354Ri/dVUGTvD+2M33O+P3XBvXr6K7vV4cHVyAusqlmJtxVKsmpjCN+q34czYEM6MDcoKqMWqlbzx+sHm5n/6Mb1hXR16+6+J8YKQseoXZ4bkEkFXVNhl28TKiXciySKto2qIsNkEYemIRmNgWRZ6vV42rHuuMQDAm51tzc6PraIBYVZyj2cSPZ5J9McDge3Lq7B9eRXOjA3hfz9ob8rV/XR1XnR3dl5019dvoSvK7fDF07PRaBS7Hn8KA/19BUuGlJaWwOOdyipvUGiRpVRVrXYCiddDSsuVy9/z7NkzJ6TtcGWUniqANADA+2M33Lte20s9Vn+v69P19zrJpAH7klL4ZhJu8dL1npzFJrz5+sHmO+t/TNttZWDCYTBMWJy9Urdmrew9YFpRWjdBq9jZNqeypaysjBY+qa+uX+j6O93vIesP/03nedeu1/ZSh7ramgHALlkV8IXzJ5vSdcxrpavzorur86I7BuFlLVK0rpKoRLlcZTYUYiiwdgSxGcVAT5PJTOdzV6V1p2viqY54+U1nm+uzr/+aOtTV1mxfUlowsQlvvn6wGQDMZrM4nZhAZqfOBb093e7Z7qwAAL/f7z579gw1NDQkC6RYNuz2eiby9prKNrhmC1fym84211cOv0IVUmwgYeVA8otSc7VyJR5P5h69Qo731sLQ0KDr7Nl3qaGhoWavZ6IpH7GvfHA55bF0ghdtJUbX7m83vfnb/4uVmM0oMZtlbk6v12VVl0vb4FqZzTo7E5MT4658rrfZbDQTmnGS9vzGOxrEBE6mJl5R11p98/WDzZ/e9XnnihXLMDCQaPYJC+rp0q4skYlM2cFiiZ0v8QmerQDw1n//FwDgw3/7iXh8Oph+5k9R33nyxmsHXWVWC4wGA+x2iWuPCW/000qqKH0hsnlzvWylaWVLJFN+pOgvuenqvOi2Wi1YuXI5SkpMkDZb9CkWG5Bys1pqLlRVrXbxPA+W5cDzUXg8XllHkZaMXdGXz35p/6+bf/rvv6RLS0qwcuUK9PVfE5er0ul04BV9tWrJkfXr6lofeOAhGgAsltvmsPTFg+d5TM/MyPZpyX4WXXAdRcFoNCIajaLcbsNoaSkCwWkhWo8LryUxEggGxZ9WiwXDGdZVvRk9g5CtU0drtF90l37hgtAtqNPpoNfrUV1dlXirb9y6+3q6T6QT6JUDLTJXRsRfaAwNDRZ3blmhaJeIrtYGz2SNFzsuJB1P59pvRusGSDv+DEWEZ5iwmLzxeiaatNxjXgieCS0CKUW33LYk5bn5ji4pNkT4XJI380Lw97suiQLYFGu9aLVGpVu/FcglSzcvBDeZTFhVuRJWqyXpWK7WuBBdeiGYF4ITrBYLVlWuxLJlS7O+9mLHBXeSW08h+qLgRWbb9h2ySV7j45PFKsqCZ14Ino5srFFLPX4rWzcwDwV//3K3bDsbgS52XHCbjIlXQam59Js9Qs+XeSH4tm076ELdq6vzottkNMJkNKZtmt2qFD21mo5c3a8xbuUl8fVcF0kwLyxcStflxIS/XNxvy77nxXpc2aZfZB4KXmgWRZdTdMG3bU/U3+MT8uZYLi79woXzbmkCRzlm7lan6HX40OAg/bnHH8Ojj34GFqtdHJsF5F6HWy0WWC0WBILBnJflXqgUXfCq1UIf76FDb8Dnm4LPNwWgMPOtrBYLPrK0QrZvsR1eRKqqVrsyvV8zX5RZu1td8KJaeFlZmSMWE96/SFGU+EI7IPfpN9KYYJFkimrhViuZcyUsvs8whXtfNmFsYkL8fKtbNzAPovTZRurSb/W0KlBEwcvKymjpGC2fZGVHlg27c3bp23bSeRduAVO0Otzv97v9fr+bbN9mKXNmu6pEJpTt+kXmQbNsaGjQBQBlZWWyL0AhWIzQk5k3dXihxL58uSvlsUXBhSUJFhTbtu+IAcCjj34GM2FenHDX29Pt1jqUdyFTdJdeSFavrnaRz4cOvYH+/mviMY5lb/kIHZhHLr0QWK1WR6rM3Wysg3YzsqAs3Goto6WZu0WSWVAWLkW5+N6ihQssGMETS2MJKNO0+a6UtFBYMIIDqWdX5pO5W2gsmDpcmbkzGE0Fz9wtBP4fRUmifHVC2s4AAAAASUVORK5CYII=", "art/refs/chicken_coop.png": "...", "art/refs/apple_tree.png": "iVBORw0KGgoAAAANSUhEUgAAAWAAAAHJCAYAAAC7cuNqAAAAAXNSR0IArs4c6QAACn9JREFUeJzt3TGPHGcdwOGxlQopokCIjgp/CiREgSxR0Zg2DUWqUEH4DEAHBUrhxi1p3EYUCIlPYapUF0UuIiSXmMJZ+7K+3ZvZndnf7O7zSGl85807K+t3/3v33dkHNzc3rwcATu5hvQCAayXAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAAJEP6gXAWB//89ejvu+zn/994ZXAPB7c3Ny8rhcBdxkb3PsIMmslwKzKXNHdRYxZEwFmFZYO7zYhZg0EmNSpw7tNiCk5BUGmju9a1sD1MgFzcmuNnmmYUzMBc1Jrje8wrHttXCYB5mTOIXDnsEYuhwBzEucUtnNaK+fNHjCLmxq0H37vR8MwDMPv//ifO7/+6ScfHr2mMewJszQBZlFj47uJ7jDsDu+2U4RYhFmSLQhyt+ML10SAWcyY6Xc7vmOn32EYhj/99b+T1zSV/WCWJMAs4pD4HuIU07MIsxQBJjFnOG1hcK4EmNndNzEuEcylI2wKZgkCDBARYGZ1yZPiJV8bDQHmpO7bKvjzH34y+rG2v9deMOfGZ8JxNXYdWzvVO+tgm3fCMZu5j57tOhO8b0r++tVX7/3Z2PPCY0Ps3XHMxQTMak3ZjoBzZA+Yizbl3XKneGcd3GYC5mhPnj0ehmEYnj6/++f5pe2xbq7384++iFfCuTMBc5Qnzx4PT58/3BnfYXgzWW6my7v2aOey5GPf9oMff38YhnchhkOZgDmI+LxhGuYYJmAmux3ffZPvtkveY/UDiUMIMJPMEZoltgpOtf2wjwgzlQAz2pyBmTOY+x5ryguAc7xYKMJMIcCMskRY5ojwGibfbSLMWALMvZYMyjEBHft3P/3kw73T7X1fP4QIM4ZTEOx1ipB8/eqryTfSOSTcpz6P/OTZY6cj2MsEzFF+86v/jf7efQH8+tVXb/875nvgnJiA2an6NfqSAmsKZh93Q+NOh8T3Wt6K/PLLbyb/HRHmLiZgZnNpoYWl2QMGiAgw73GEan6eU+4iwMzmkL3Rc3MN18jpCDBARID5Dr8qL8dzyzanIJjVyy+/eXvD8kv7FGLbD8xNgJndfff93Xz9XEMMc7EFASOYflmCADMrn5AB49mCgC13/2B4OOnGQzCGAMO37pvIN9O9EDMXAeZsXdopC66PAHN2ljhlMWU/+ulz2xHMw4twABEB5juOvW/tXJ+QscuUSXVtpyzcE5htAgwQsQfM7DZT8LV8QgYcSoB5z+cffTHLjWN2bkd8+66yzT0jam/e5bbsL4O2H7iLLQgya3h772YNU/aunYBgLiZgUi+jaXgN8QefisxOxf1rx4R47OmGu/aax4R31971oZOv7Qd2MQGz01x7wVPcDuRcU/HUaXfOLQbxZR8BZrW2w7kJ8may3TUJvw2obQZWzhYE91rrR+lspsu1rw92cQqCe60xJLfXtPb1wS4mYEZby6T5s3/fvUf7r5+uY54QX8Zax79YzsIawrIrvvd97VTW8BxxPgSYScrAjAlsGWHxZSoBZrIiNFPCWkRYfDmEPWCOsvS+8KEx/e1n/xiGYfn1CS/HMAFzlCUDNMcku+T6xJdjmYCZ1bET5+2o/eXjXxz8OJsJeNuc64NjCTCLGRu7fVE7NMK7AnzbHOuDYwgwq3dIhMcEGGr2gAEiAszqmWa5VAIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCDyQb0AePHiRfKYjx49mv3/C1MIMIklonvMGsSYggBzEmsI7j7b6xNkTkGAWczao7uP6ZhT8CIcizjn+G67pGthXUzAzOpSY7W5LtMwcxJgZnGp4d0mxMzJFgRHu5b43naN18z8BJijXHOIrvnamYcAA0QEmIOZAD0HHMeLcJyFX/7ub/USYHYmYA5i8nvHc8GhBBggIsAcxDnYdzwXHEqAASICzMFMfp4DjiPAABEB5ijXPAFe87UzDwHmaNcYomu8Zub34Obm5nW9CC7HpZ+JFV7mJMAs4tJCLLwswRYEi7ikYF3StbAuJmBO4twmYtHlFASYxNqCLLgUBJhcFWPRpSbAnIWpkRZXzoEAA0ScggCICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggIgAA0QEGCAiwAARAQaICDBARIABIgIMEBFggMj/AW9iQ6NY61S6AAAAAElFTkSuQmCC", "art/refs3d/oak_tree.png": "...", "art/refs/crop_strawberries_sprout.png": "iVBORw0KGgoAAAANSUhEUgAAAWAAAAHJCAYAAAC7cuNqAAAAAXNSR0IArs4c6QAADNJJREFUeJzt3b2S29Ydh+EjTeSx5MqZVBndkUs3bnIdadXmOtK4cZk72slskYkrSx6rcAqFtoxdkiB4gN/5eJ7W2gP8ycU7MAhiXzw+Pv5aADjcy/QOAMxKgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESAAUIEGCBEgAFCBBggRIABQgQYIESA6dbDw0N5eHhI7wZs9qf0DsCtTtH91z/+Vkop5Zu//7OUUsrbt29j+wRbvHh8fPw1vROwxjK8f/7qi1JKKf/96ZdSihDTHwGmeefCuyTE9EaAadba8C4JMb0QYJqzNbxLQkzrBJhm1ArvkhDTKgEmbq/wLgkxrRFgYo4K75IQ0woB5nCp8C4JMWkCzGFaCe+SEJMiwOyu1fAuCTFHE2B200t4l4SYowgw1Z3C+8O770oppfz1L18ld2czIWZvAkw1n5/x/vzhY/n6zas//PfXb/o4A14SYvYiwNzt3KWGD+9/efbfCzF8IsBstvYarxDD8wSYm239cE2I4Y8EmNVq3dUgxPCJAHPVXreTCTGzE2DOOuo+XiFmVgLME6kvUAgxsxFgftPKN9eEmFkIMM2Ed0mIGZ0AT6zV8C4JMaMS4An1Et4lIWY0AjyRUR6SI8SMQoAnMOpDcoSY3gnwwC5danguXr2GS4jplQAP6JZrvELcLiEenwAP5J4P14S4XUI8LgEeQM27GoS4XUI8HgHu2J63kwlxu4R4HALcoSPv4xXidglx/wS4I8kvUCRCfNpm7e0IMa0Q4A609M21I0J8VCCFmDQBblhL4V3aK8TnorhlG2vPoIWYFAFuUMvhXaoZ4mvxXbv+1qAKMUcT4Ib0FN6lGiGuEeAaZ9BCzFEEuAHLh+R8/eZVtwf7PSG+N8C1zqCvrdfreyPE7RHgoOUZ7+sXT/9Nrwf7lhC3FuBr6/b63ghxOwQ44NqlhtHvvd0roHsF+Nr6vb43QpwnwAe69RrvjCG+5xru3gG+tp1e3xshzhHgA9z74dpsIa59F8Ot66wlxNxLgHdU+66G2UJca93a21i7zV7fGyE+jgDvYO/byYT49jVrrNvqdvcixPsT4IqOvo9XiNskxKwlwBWkv0AxerxGmqWUfucR4voE+A7p8C6NHq+RZiml33mEuB4B3qC18C6NHq+RZiml33mE+H4CfIPWw7s0erxGmqWUfucR4u0EeIWHh4fyw7vvypevX5VS2g/v0ujxGmmWUvqdR4hvJ8AXPPeQnFL6PUBGj9dIs5TS7zxCvJ4AP2PNQ3JK6fcAGT1eI81SSr/zCPF1AvyZLQ/JKaXfA2T0eI00Syn9ziPE5wlwqfOQnFL6PUBGj9dIs5TS7zxC/NTUAd7jITml9HuAjB6vkWYppd95hPh3Uwb4iIfklNLvATJ6vEaapZR+5xHiyQKceEhOKf0eIKPHa6RZSul3nplDPEWAW3hITin9HiCjx2ukWUrpd54ZQzx0gNPfXGvhAFnuwz3bHj1eezyP+IjXp4Xfs5pmCvGQAU6HdylxgFx6MPkez9vt9WCvOcuer/k92+/1vZkhxEMFuLXwLh15gNT6yxCXzuiE+PLPb1mrxhm0EPdjiAC3Ht6lvQ+QGn8b7ZazudlDXOtv0dU+gxbi9nUd4N7Cu7TXAbJ3gM/97KwhTr3eawlxu16md2Bmr998cTZkaw/qPazZ9rlAPXd2nJxlqyNn2fp6r9Xq7xmdnwGf9H4mfFLrTOXeM7I9/5e617OuS7O08nqv1esZ8UhnvidDBPhEiK+vsWat2kGYIcQtvd5r9RLiEcN7MlSAT4T4vg909grC6CG+pMUAX9t++r0ZObwnQwb4RIi339Z09IdC6YN9q71uP9u6zj1aCfEM4T0ZOsAnQlxvWzW3N3qIa167T3578qh9mCm8J1ME+ESI79/WUbGfNcSJrzKfc9Tv2YzhPZkqwCdC3CYhbtNev2czh/dkygCfCHGbRo/XSLOUcvs8wvu7qQN8IsRtGj1eI81SyvV5hPcpAf6MELdp9HiNNEspT+cR3vME+BlC3KbR4zXSLKWU8uH/ZRHe8wT4AiFu0+jx6n2WH99/LKWU8u2770spwnuJAK8gxG0aMV6f622W06WGnz98LN+++154VxDgGwhxm0aI10mPs7jGu50AbyDEbeoxXuf0MIvw3k+A7yDEbeohXmu1OIvw1iPAFQhxm1qM11YtzCK89QlwRULcphbiVUtiFuHdjwDvQIjbJMS3Ed79CfCOhLhNQnyZ8B5HgA8gxP1sZ+YQC+/xBPhAQrx97ZrbWLPNmUIsvDkCHCDEt6136/pb92u2EAtvngAHCfG6dW5Zt8YZ9Ogh9pCcdghwA4T48s+vXa/2XxceLcQektMeAW7I7CFuLcCX1u0pxC41tEuAGzRriFsN8KX1Ww6x8LZPgBs2Y4jvuYa7d4AvbaelEAtvPwS4A7OFOHUJ41athVh4+yPAHZk1xLdEbcb7iIW3XwLcodlC3Mq6W7a75zaFt38C3DEhXrfu0ZcF9g6x8I5DgAcgxG2qHWLhHY8AD0SI23RviIV3XAI8ICFu060hFt7xCfDAhLhN10IsvPMQ4AkIcZuW8/z4/mP58vUr4Z2IAE9EiNv07//8VErxkJwZCfCEhLgNLjUgwBMT4gzh5USAEeKDCC9LAsxvhHgfwss5AswTQlyH8HKNAHOWEG8jvKwlwFwlxOsIL7cSYFYT4ucJL1sJMDcT4k+El3sJMJvNGmLhpRYB5m6zhFh4qU2AqWbUEHtIDnsRYKobJcQeksPeBJjd9Bpilxo4igCzu15CLLwcTYA5TKshFl5SBJjDtRJi4SVNgIlJhVh4aYUAE3dUiIWX1ggwzdgrxMJLqwSY5tQKsfDSOgGmWVtDLLz0QoBp3toQCy+9EWC6cS7EwkuvBJjuLEMsvPRKgOnWKcTCS68EGCDkZXoHAGYlwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAh/wM51kbVDQfWGQAAAABJRU5ErkJggg==", "art/refs/crop_strawberries_growing.png": "iVBORw0KGgoAAAANSUhEUgAAAWAAAAHJCAYAAAC7cuNqAAAAAXNSR0IArs4c6QAAD5RJREFUeJzt3TGoJddhx+GzTtZ415FAwaUKFa5SpBAOLgwmhQpBQIWNkBrVqlOlMiykUuU6dRqHsC4MgoWoEA4pDKpSqErh4hK2WCJQol2jRTjF86zu3r1v38ydM+d/Zub7qn0r7eXeeff+3nlnzpy59fDhwz8WAJr7TvoJAOyVAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwQIgAA4QIMECIAAOECDBAiAADhAgwq/Xhp++WDz99N/004GK3Hj58+Mf0k4Axxsb2n/72Xxd+JlCHANO9S0e5QkzvTEHQtTlTDKYn6J0A060aARVheibAACECTJdqjlyNgumVAAOECDBAiAADhAgwXaq5htd6YHolwAAhAkx3DodDORwO5c37j2Y/1pv3Hz17vIE9JOiFS5HpxhDJjz/6oJRSyl9+/7vlk7dfbf48TFnQigATdy68p4SYLRJgYsaE95QQsyUCTHOXhPfY7376/Rf+7n/v/tn8J/YSIswSBJhm5oZ3cBrgpeM7EGFq+/P0E2D7bgrvv/zV1Rjgvc9vtX1iEGYEzGJOw/tvf3P7osc5DfPxCLjV6HdgFExNRsBUN4T3/r33Syml/PtP7856PCNktkqAqeZ4xPuHJ09nh/fUEOI3qj7qNIfDobz++uvBZ8CWmIJgtnNzvEMsl/LGD/7i2Z9bTUMMV+b93T/8cymlCDGzCTAXu+7k2tLxHQwRbhXgtx58WUop5X+++rqUIsTMJ8BMNnZVw9JajoKH+B4TYuYSYEYbs463VXwHLSJ8Lr7HhJhLCfCOXbcj2OlSqykXUCQD/OPfflVKqXe58k3hPXVTiMceb/ZDgHdmyjaMv/jhL0sp065c6yHAx6bGeGp0zzkN8T/+19+P/rdivC8CvCOX7oE7NUqJOeBSzgc46dKRuAjvhw3Zd2LOBuSJHcjWbs4xs1n8fgjwDtT4QE8JSosr1k5Hvz2p8QNLhPdBgFnEkhHuOb4whQBvXM2R1NSR3Xuf36oa4nc+e1re+exptcdbQs3pGqPg7bMXBIs7jvDUE3TPBfzuZfsHQ68EmKaWnJp48vhq+dcdoWYlTEFs2PGt2PfkyeOvn8V47fb6PdwLAd6gw+FQDodD+fijD57t4FVDjYsUWkqEuOYxevP+o/LxRx88+36yPaYgNqTWPdfWaph6OI3uWqcmhu/f8P2018T2uBJuA8aEd+7Z+WFkd9PjtBgln7srcikvXgl33ei3RYhbHW9Xza2bAK/Y1BFv4oq2JYI8NsCDVIgTx1uQ10WAV2jOVEPqsuKaIZ4a4IEQ0xsBXpHac7ytw1ArwpcGeLCXEItw/5yEW4GlTq698vib575e+q4Sn7z9ahcrKVIn606PdynLHvPhSjoh7pcAd2yLqxp6iXAp+VUTre5lR78EuEM170Ax9sqzPcdgSohrHe+WPvz03fKLH/7S8rUOCXBHlrjn2un/30MgehoFHzsX4t/86HYpZfwxH3u8W//A+/ijD6wj7pCTcB0Ywnv/3vullFJeu3v7hV9/a99l4r3Pbz13Mqt1EHoM8LEtHm83D+2PAAedjnjvnBks3bn73cVu8dPytu7H9hbfwek+xqljLsT9EOCAm6Yahl+Br379XVYiwj0HuMX97IZjnj7eQpxnDrihsSfXvp1+WD4Gv3/0f03vMPGTXz8qT/705972Zmh9R+dXHn8TPflpr4k8I+AGLllO1jIGrX41fuvBl2cvguglxKljvuTxnsKIuD0BXtCcdbytR2PHQfjxb7+qftXWaQx6C/HWj/cUQtyOKYgFbOECirG7cY19nFPnlnyNuQCih93YlrD08Z7C1EQ7RsAV1QxvcgriZXsqLBXAm0bEU8N0yfNIHfPE8Z7CiHg5AlzBUiPeVkGYEuClnYb4P372g1mPNyVQezzeUwhxfaYgZtjCVEPLFRBjXHdZcAvvfX6r+VzwmpiaqM8I+AKtw9vqwoBBLyOyWiempv6avtfjPZUR8XxGwBOkRrxLjMyuLo2t+pCb0fp4u2fdfhkBj3A4HMr9e++X7925ujItOdUwNwzvfPa0lHL1YZ+7sfmSelmWNfd4H2/Gc93x/usHXzz39dpCPDAins4I+CVON8k5t1dDa6e7a03ZHvHFJV/ng8C35hzvSxkR74cAn3E61fDa3ef3ZOjpAzLlA588wbUVS2znmd4YvjYhHk+Aj4zdJOf067V9QNb2fPdCiPdHgMv0TXK28gG5zpPHX8dfy1sPvoytgkjb2vtMiK+36wBfuqphax+Qc7yWvK29z4T4RbtcBVF7OVnqNudzjT0rX0rutbS+lfuSo+W5q07W+j67jlUTOwvw0ut41/YBuSkIvexY1jrCg9oxrrXsb23vs5vsOcS7CHDrCyjW8gEZGwQhrhPi2uuu1/I+G2uPId50gNN7NfT+AZkahF5CPDzvYSPzFneWqBHhpS586f19NtWeQrzJk3Dp8A62dhLl0j18l/LK429KKW3urfbJ2692u5pia++zPZ2s21SAewnvqa19QHoLMVe29j7bQ4g3EeBew3tqax+QS0I8dW+Fm648a3lTy55Hwce29j7bcohXHeC1hPfU1j4gN4V4zoY2x/92icuAt2xr77MthnjVAV679Aek9q5np6/nNz+6XUqpd453iPEb1R5xujlXCaZ2mUu/z7jeJlZBrHUkfGpLZ7OXvrNEi9u6n/rJrx899/Uavy+lrPd9tsXVEZsI8ECI+9D63mqtAvzWgy+7WYpXw1reZ1sM72BTAR4IcVbi5pat1wEL8fK2HN7BJgM8EOL2Wt/UskWEX7byQYjr20N4B5sO8ECI20kGeDjJldjGUojn21N4B7sI8ECIl9dDgAeXhnjOWl8hnm6P4R3sKsADIV5WYg64lPHLvIYwL3lRhRDfbM/hHewywAMhXkbvAW5JiF8kvN/adYAHQlxfy3XAgx4DPBBi4T1HgI8IcV1LRfhcfEvpO8CDPYZYeK8nwGcIcV21QvzOZ09LKaX859uvnf3vawjwYA8hfvKnb7vwXk+AX0KI67o0xKeb8Cy1sXnCFkP8xeOrH5Q/v/erUorwvowAjyDEyxrCPHa3s5fdTDT9Wi61hRAPUw1/ePK0/Pzer4R3BAGeQIj7MOZuzmt5LafWGGJzvJcT4AsIcdaYAA96fy3XWUOIhXc+AZ5BiDNumgNeQ7zG6vG1CG89AlyBELc19iRcj/G6VA+vRXjrE+CKhLiNqasgeohXLYnXIrzLEeAFCPGyLl2GJsTTCO/yBHhBQryMueuAhfjlhLcdAW5AiOuqdSGGED9PeNsT4IaEuI7aV8LtPcTCmyPAAUI8z1KXIu8txMKbJ8BBQnyZpfeC2HqIbZLTDwHugBBP02oznq2F2CY5/RHgjghxn9YeYlMN/RLgDglxn9YWYuHtnwB3TIj71HuIhXc9BHgFhLhPvYVYeNdHgFdEiPuUDrHwrpcAr5AQ96l1iIV3/QR4xYS4T0uHWHi3Q4A3QIj7VDvEwrs9ArwhQtynuSEW3u0S4A0S4j5NDbHwbp8Ab5gQ9+mmEAvvfgjwDghxn05fzxePn5bv3bktvDsiwDsixH3670dXmwjZJGd/BHiHhLgPphoQ4B0T4gzhZSDACHEjwsspAeYZIV6G8HIdAeYFQlyH8HITAeZaQnwZ4WUsAeZGQjyO8DKVADOaEJ8nvFxKgJlMiK8IL3MJMBfba4iFl1oEmNn2EmLhpTYBppqthtgmOSxFgKluKyG2SQ5LE2AWs9YQm2qgFQFmcWsJsfDSmgDTTK8hFl5SBJjmegmx8JImwMSkQiy89EKAiWsVYuGlNwJMN5YKsfDSKwGmO7VCLLz0ToDp1qUhFl7WQoDp3tgQCy9rI8CsxnUhFl7WSoBZndMQCy9rJcCs1hBi4WWtBBgg5DvpJwCwVwIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMEPL/HcS5kDcSWxkAAAAASUVORK5CYII=", "art/refs/crop_strawberries_ripe.png": "iVBORw0KGgoAAAANSUhEUgAAAWAAAAHJCAYAAAC7cuNqAAAAAXNSR0IArs4c6QAAEWJJREFUeJzt3bGLHFeCx/E3ZmVWMivj1UW3Qn+AYmeXnsHgYIMNtBgcbbyxU6eOHTsyGCvYgzMYDuy/Q3+AmD0UrHaRF0uHHcwF2pJaNV3dVdVV9XtV/fmAwR4Lpudp5juvX716dfHkyZOrAsDi3kq/AIBzJcAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwKzW5eVluby8TL8MGO1X6RcAQzXR/e7zT0oppXz06VellFLu3r0be00wxsWTJ0+u0i8C+miH97fvvF1KKeXvP/1cShFi1keAqV5XeNuEmLURYKrVN7xtQsxaCDDVGRveNiGmdgJMNaYKb5sQUysBJm5seH/48HYppZT//J8fe/15IaY2AkxM3/A2oR2qK8xCTC0EmMX1Ce+x6H7x5Zvftn/+08XBP78vxkJMmgCzmDnC2ybErIkAM7uplhqOxbcxJsKlCDHLE2BmM+Ti2lTxbYyNcClCzHKcBcHkmvD+5bM/llJK+fd/e6fzzz68f1Xu3Ht3kde164cPb5enj5+VB4+uh7r5ReGsCeYmwExmd8b7fy9+Ke/dulFKKeXF85czypu33pwBp+LbuHPv3fKw7I9wKULM/ASYk+1danjn7VfhbeyGOB3fxrEIlyLEzEeAGe3YGm8z422H+OH9+i47PLx/dTDCpQgx0xNgBht659puiL99/8a8L65lyH7hPhEuRYiZjl0Q9HbqWQ27M9+hyw9Dd0Ec0w7x08fPXv17nwjvsmuCsQSYo6Y4JKe97DBm/XfOCO8GuJThES5FiBlOgOk01elk+9Z8x16AmyvC7QCXMi7CpQgx/Qkw10x9LGTXRbfdCA8926Hrz4+5YWNffEsZH+CGEHOMi3C8Mtd5vId0BbP5eFeIjwW6Bi7WcYwAEwlvKf1mq198ebWK2B4ixHSxBHHGlghv1/LD198PO+O3b4SHLkF8/MHhw9xPXYbYx9IEDTPgM5Sa8fKSGTENAT4jQw7JYX5CjCWIM9B1SE6jfUjO1PYtQ8y1BFHKsGWIQ0sQcyw/HGJp4vyYAW9Y1yE5pbx5PkPXaWVr9ec/XfSOcPOL4Nha8BLMiM+PGfAGDVnjbR+UU8r0IV56BrzrlNnw0jPgNjPi7RPgDTnl4trcIW5HeIkAP338bPDnaSKcju8uId4uSxAbMMWuhn1HR865NPHxBz/2juPHH/xYyuPJX8JqWJrYLjPgFZtzO9kcM+IxSxGnrs2OmQHXNPvdx4x4OwR4haYI77GHYDYPrZw6xH1vzJjqotjQAP/37/45yedt6zveQwjx+gnwipwS3mMBOOY//utv1z42NsRLPhEjFeBTx3tIkIV4vQR4BcaGd+yj3g9d8JoqxEtFeGiAv3nv6Ukz/ENjPma8S+kfYyFeHwGu2NLh3dVn10E7xkPDtUSExwS40ffr6TPbnWLMhXh7BLhCcy819N0bOybCpdQX4r4R3o3vrkNfz9LjbWliWwS4IqdeXJsyBo2+UZjiYt2cIT4W4d21331fSynXv57keA8hxPUS4Aq0D8l579aNQfF6eP+q9yN+5ghCKdPumpgrxA8eXZTf//U3b3zs0EW3QyHue5FtrvF++viZh4dugBsxgtoz3t1DcvreBDEkvnP64cPb/4rC6Td0tMMyNsj7AjVkl8O+m1O+ff9GuXPvtB0OU7hz793ysAyLsBs66iPAAceWGvrGa8ntXH09vH9VHjy6mPTOuvSNEc3rrXm8hxDiegjwgvqu8faJV40xaOxGYelbnOeylvEeQojzBHgBYy+udcXr2/dfL1UssfwwZv9qOwprDvHS8e073nfuvfvqic5jI1yKECe5CDejqc9qaMe3lHEBHnph6JB2FNqPeO+KwhLHYE6hHd85x7vvOca7Y953vIdwsW45b6VfwBZdXl6Wy8vL8t3nn5TvPv+k/Padtyc5KGfJQM31EMzGzVtvX/t6Xjz/uXPnwZrNPZZTa75fm+/f5vuZ6ZkBT2juh13ueyt8yhLEobe6Y7ZPtWdjjT6zshpnxF1LD2PHfOrxLuX6DLiU6S9amhHPxxrwBJJPGX76+NmrIAxdqx37lIk5rHGNOD3eXb/wpmaNeD4CfIJaHu9+bPb0xZdXVcX2kLWE+NCYN/9vLWPelxBPT4BHSIT31Cvxa4pwKXWHuJa12i6n7IjoQ4in4yLcAHNdXDvFkNO+5gzHobfDp/zySF2s63tw/CGp8V6Ki3WnMwPu4fLysvzlsz+WX998uQUsHd1zVPOM+NyZEY9nBnxA8xu9OSTn5sXLf87BkOWKY48PmvLt8FIz4qVvf55yvFPMiIczA97j0CE5pWRmXg8eXVR7O2zztnzJMGxxRtx3O9qx8U6fnWFG3J8A7xhySM7uf6/1B/6YZlbWdy3z6+9vLz4721qIh+wJTvziG0KIjxPgctohObv/vcYf+D6amyz6XIBKRLiUbYV4yHivgRB3O+s74U7dTtb3yQlT2rcM0fcHdTeMx+7mal9lHxqD5nOl3g5PdWed8Z6eO+teO8sZ8FT7eNc0I27PSmvYxjSnOWfEH3/w4+BAbn28hzAjfu2sZsBz30Cx1Ix46B7VKZYExszIapqNnTIjNt7zOucZ8VkEeOk715YI8dI7IoYGYcijf47p+/y1tn0Prxwb4trHe/eJzjW98xriHEO86QCnz2qYO8RLRiER4EPhbe8UOLSPdqoQ1zzeuwFuCHH9NhngdHjb5gzxUlFYMsBDwts2d4hrHu/EReE5nUOINxXg2sLbNtcPSG1RGBvfY0sNQ54scchaQjx2vIV4PTYR4NrD27bWGXGfIDRvhYd+LVPFtzEmwqXUF+I+Y971C0+I67fqAK8tvG1bC/HYdcg+F9mmDnAp3REupa4Qd41333caQlyvs9wHXIs59xE3W5KmDsODRxedP/g3y7C9tw/vXy3yVOcuP3x4uzx9/Gzv9q2h+4gT493Xmvarn5tVz4Aba58JN5acqRwLxRR7Sg/NIofGd44ZcKMrwrtOvbNuifHua60z4i3NfBubCHBDiOvU/nq+ff/l6XI1BbiU9T48dKy1fJ9tMbyNTS1BNH8xzV/UWkO8tbeMu19PE9+lDNkv3OdRPls69Kf277Mth7exqRlwmxlxXXbfhg9d+5368T7tEO+e1TBkOcCMeHrnEN7GpgPcEOK89hromItvc0a4fVjO0DVZIT7dOYW3cRYBbghxzhQBLmW+CJ8a4IYQD3eO4W2cVYAbQrysrh0AuxEeslZ76M+PuVjXdVTkKTsThPi4cw5vY1MX4fpysa4eXcFsPt4V4iG7HBJcrOsmvK+dZYAbQpzVZ7b6xZdX1cf2ECF+TXivO8sliC6WJqY39DDzLn0jPHQJ4tjh6VPfIHGOSxPC2+2sZ8BtZsTM7ZxmxC/+9btQeLsJ8B5CzNy2HOJ/PP+llFLKHz77ppQivIe8lX4BNbt79265e/du+ejTr8pHn35V/v7Tz6/eTq3JzVtv7/2hfvH85863kVNZ+jllU64XL/Ha9/3dLPH3MocXV69nvX/47JtXPz90swY8gDXicU55tHtjaFinWAtOPORyjWvE1njHE+ARhHiYRIAbQ0LcjnDyKcNrCLHwns4a8AjWiNfh5Q0W/UP/9fe3J3mk/BRqXiMW3umYAU/AjPi4U2bBXVFs7qTrupNtyOdof67k7HefGmbEwjs9AZ6QEB82JsKnzkjHBLi2+O5KhFh45yPAMxDibn1vzJhqKWDM493XYIkQC+/8BHhGQrzfEo90b2w1wI05Qiy8yxHgBQjxdUtFeOsBbkwRYuFdngAvSIjftESEhwb4m/eellLyOw3GGhNi4c0R4AAhfq2mCDfx3bXlEAtvngAHCfGb5ozxsQjvLj3UsOVrKvu+Fofk1EOAKyDEb5orxA8eXZTf//U3b3zs0Jrv1kLskJz6CHBFhPi6U2M8xZ7etYfYUkO9BLhCQtxPE+elbpxYW4iFt34CXDEhrlPtIRbe9RDgFRDiOtUWYuFdHwFeESGuUzrEwrteArxCQlynpUMsvOsnwCsmxHWaO8TCux0CvAFCXKepQyy82yPAGyLEdTo1xMK7XQK8QUJcp6EhFt7tE+ANE+I6HQux8J4PAT4DQlyn9tfzj+e/lF/fvCG8Z0SAz4gQ1+l///ZTKcUhOedIgM+QENfBUgMCfMaEOEN4aQgwQrwQ4aVNgHlFiOchvHQRYK4R4mkIL8cIMJ2EeBzhpS8B5igh7kd4GUqA6U2I9xNexhJgBhPil4SXUwkwo51riIWXqQgwJzuXEAsvUxNgJrPVEDskh7kIMJPbSogdksPcBJjZrDXElhpYigAzu7WEWHhZmgCzmFpDLLykCDCLqyXEwkuaABOTCrHwUgsBJm6pEAsvtRFgqjFXiIWXWgkw1ZkqxMJL7QSYao0NsfCyFgJM9fqGWHhZGwFmNbpCLLyslQCzOu0QCy9rJcCsVhNi4WWtBBgg5K30CwA4VwIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMECLAACECDBAiwAAhAgwQIsAAIQIMEPL/+kKsYJl5UrMAAAAASUVORK5CYII=", "art/refs/hay_bale.png": "iVBORw0KGgoAAAANSUhEUgAAAWAAAAHJCAYAAAC7cuNqAAAAAXNSR0IArs4c6QAABttJREFUeJzt3bGKU0EUgOHJ4nMsNqlWfLV9Fl9NtEpnddttBV0LWUSssnLnvzv5vj65ZyD8nJCBnLZtex4ATHdXDwBwqwQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYIPKuHgB+fvtUj3CVu/vHegQWYQMmJb7cMgEmI77cOgEmIb4gwATEF34TYKYSX/hDgJlGfOFvAswU4gv/cg+YKX58f7r6NV8/fxljjPHw8cN/Pfs172MzYQafM4CIAANEBBggIsAAEQEGiAgwQESAASICDBARYICIAANEBBggIsAAEQEGiAgwQESAASICDBARYICIAANEBBgg4j/h2NXlchljjPH+FA9ypZe5z+dzPAkrswGzm5eIvWUrnIHjOm3b9lwPwZpWiZctmL3YgNnFKvEdY62zcCwCDBARYHax0tf2lc7CsQgwQESA2c0Km+MKZ+C43IJgirf2Q5bwMoMAM9XRQyy8zCTAJI4WYuGlIMAcxqwoiy1HIcAc3mvDLLQcnQADRFxDA4gIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQEWCAiAADRAQYICLAABEBBogIMEBEgAEiAgwQ+QWEj2XlD2auHgAAAABJRU5ErkJggg=="}''')
os.makedirs("/content/farm-art", exist_ok=True)
%cd /content/farm-art
for path, data in EMBEDDED.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "wb").write(base64.b64decode(data))
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
BFL_KEY = secret("BFL_API_KEY")
manifest = json.load(open("art/manifest.json"))
items = [i for i in manifest["items"] if not ONLY or i["id"] in ONLY]
print(len(items), "items:", [i["id"] for i in items])

In [ ]:
# 4. Pick and load the model
import base64, io, time, requests, torch
from PIL import Image
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if BACKEND == "auto":
    if gpu_gb >= 35 and os.environ.get("HF_TOKEN"):
        BACKEND = "flux3"
    elif gpu_gb >= 14:
        BACKEND = "flux2"
    else:
        BACKEND = "api"
print(f"GPU: {gpu_gb:.0f} GB -> backend: {BACKEND}")

if BACKEND == "flux3":
    # FLUX 3 Image Klein (distilled: 16 steps, no guidance). The 9B prompt upsampler is switched
    # off: the manifest already gives full prompts, and it would need another 28 GB.
    from diffusers import ModularPipeline
    pipe = ModularPipeline.from_pretrained("black-forest-labs/flux-3-image-klein")
    try:
        pipe.blocks.sub_blocks.pop("prompt_upsampler")
        pipe = pipe.blocks.init_pipeline("black-forest-labs/flux-3-image-klein")
    except Exception as e:
        print("Upsampler not removed:", e)
    pipe.load_components(dtype={"vae": torch.float32, "default": torch.bfloat16})
    pipe.to("cuda")

    def generate(prompt, ref_paths, seed, width, height):
        refs = [Image.open(p).convert("RGB") for p in ref_paths]
        args = dict(prompt=prompt, image=refs, generator=torch.Generator().manual_seed(seed), output="images")
        for extra in ({"height": height, "width": width}, {}):
            try:
                return pipe(**args, **extra)[0]
            except TypeError:
                continue
        raise RuntimeError("generation failed")

elif BACKEND == "flux2":
    # FLUX.2 [klein] 4B, Apache 2.0. About 13 GB with CPU offload: fits an L4 or T4.
    from diffusers import Flux2KleinPipeline
    pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()

    def generate(prompt, ref_paths, seed, width, height):
        refs = [Image.open(p).convert("RGB") for p in ref_paths]
        return pipe(prompt=prompt, image=refs if len(refs) > 1 else refs[0], height=height, width=width,
                    guidance_scale=1.0, num_inference_steps=4,
                    generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]

else:
    # BFL's API: POST the job, then poll the polling_url until the image is Ready.
    assert BFL_KEY, "No GPU and no BFL_API_KEY secret: switch the runtime to a GPU, or add the key."
    API = "https://api.bfl.ai/v1/flux-3-image-klein"
    HEAD = {"x-key": BFL_KEY, "accept": "application/json", "Content-Type": "application/json"}

    def generate(prompt, ref_paths, seed, width, height):
        body = {"prompt": prompt, "images": [base64.b64encode(open(p, "rb").read()).decode() for p in ref_paths],
                "seed": seed, "width": width, "height": height, "output_format": "png"}
        for attempt in range(6):
            r = requests.post(API, headers=HEAD, json=body, timeout=60)
            if r.status_code == 429:
                time.sleep(2 ** attempt); continue
            if r.status_code != 200:
                raise RuntimeError(f"{r.status_code}: {r.text[:500]}")
            poll = r.json()["polling_url"]
            break
        else:
            raise RuntimeError("rate limited")
        while True:
            time.sleep(0.8)
            res = requests.get(poll, headers=HEAD, timeout=60).json()
            if res.get("status") == "Ready":
                return Image.open(io.BytesIO(requests.get(res["result"]["sample"], timeout=120).content))
            if res.get("status") in ("Error", "Failed", "Content Moderated", "Request Moderated"):
                raise RuntimeError(f"{res.get('status')}: {res}")
print("ready")

In [ ]:
# 5. Generate (on a GPU one at a time; through the API, 8 at once)
from concurrent.futures import ThreadPoolExecutor, as_completed
os.makedirs(f"{OUT}/raw", exist_ok=True)
jobs = []
for style in STYLES:
    for item in items:
        prompt = f"{manifest['common']} Subject: {item['subject']}. {manifest['styles'][style]}"
        refs = ["art/" + r for r in item["refs"]]
        w, h = (1024, 1024) if item["tiles"] == 1 else (1536, 1280)
        for v in range(VARIANTS):
            jobs.append((style, item["id"], v, prompt, refs, w, h))

def run(job):
    style, item_id, v, prompt, refs, w, h = job
    out = f"{OUT}/raw/{style}__{item_id}__{v}.png"
    if os.path.exists(out):
        return out
    generate(prompt, refs, SEED + v, w, h).save(out)
    return out

failed = []
with ThreadPoolExecutor(max_workers=8 if BACKEND == "api" else 1) as pool:
    futures = {pool.submit(run, j): j for j in jobs}
    for f in as_completed(futures):
        try:
            print("made", f.result())
        except Exception as e:
            failed.append((futures[f][:3], str(e)))
            print("FAILED", futures[f][:3], e)
print(len(jobs) - len(failed), "of", len(jobs), "made")

In [ ]:
# 6. Cut out, size to the game grid and record each image's anchor (the ground centre).
# At 2x, a tile is 256 px wide; the footprint's bottom corner sits at the image bottom,
# so the ground centre is 64 px per tile above it.
from rembg import remove, new_session
session = new_session("birefnet-general")
TILE = 256
meta = {}
for f in sorted(os.listdir(f"{OUT}/raw")):
    style, item_id, v = f[:-4].split("__")
    item = next(i for i in items if i["id"] == item_id)
    cut = remove(Image.open(f"{OUT}/raw/{f}").convert("RGB"), session=session)
    cut = cut.crop(cut.getbbox())
    target_w = int(TILE * item["tiles"] * 0.95)
    scale = target_w / cut.width
    cut = cut.resize((target_w, max(1, int(cut.height * scale))), Image.LANCZOS)
    os.makedirs(f"{OUT}/{style}", exist_ok=True)
    name = f"{item_id}__{v}"
    cut.save(f"{OUT}/{style}/{name}.png")
    meta.setdefault(style, {})[name] = {"anchor": [cut.width / 2, cut.height - 64 * item["tiles"]], "scale": 0.5}
for style, m in meta.items():
    json.dump(m, open(f"{OUT}/{style}/sprites.json", "w"), indent=1)
print("cut", sum(len(m) for m in meta.values()))

In [ ]:
# 7. One contact sheet per style, on grass, to compare at a glance
from PIL import ImageDraw
for style in meta:
    names = sorted(meta[style])
    cell = 360; cols = 4; rows = (len(names) + cols - 1) // cols
    sheet = Image.new("RGBA", (cell * cols, cell * rows), (143, 207, 95, 255))
    d = ImageDraw.Draw(sheet)
    for k, n in enumerate(names):
        im = Image.open(f"{OUT}/{style}/{n}.png")
        im.thumbnail((cell - 30, cell - 40))
        x = (k % cols) * cell + (cell - im.width) // 2
        y = (k // cols) * cell + (cell - 30 - im.height) // 2
        sheet.alpha_composite(im, (x, y))
        d.text(((k % cols) * cell + 8, (k // cols) * cell + cell - 22), n, fill=(30, 30, 30, 255))
    sheet.save(f"{OUT}/sheet_{style}.png")
    display(sheet.convert("RGB").resize((sheet.width // 2, sheet.height // 2)))

In [ ]:
# 8. Zip the results and download them (attach the zip in the chat with Claude)
import shutil
from google.colab import files
shutil.rmtree(f"{OUT}/raw", ignore_errors=True)
shutil.make_archive("/content/farm_art_pilot", "zip", ".", OUT)
files.download("/content/farm_art_pilot.zip")
print("downloading farm_art_pilot.zip")

## After the pilot
Attach `farm_art_pilot.zip` in the chat. Once a style is picked, Claude updates this notebook to make the full set in that style.